# SAADE v4 — генерація text→TTS: окремий контрольований етап

Генерація вимагає **справжньої моделі українського TTS**, погодженого використання текстів/аудіо та розділених голосів між split. Тільки після генерації та SHA-реєстрації модель може використовувати ці файли в експерименті. Тут немає вигаданих синтетичних голосів.

In [ ]:
from pathlib import Path
import sys,pandas as pd
sys.path.insert(0,str(Path.cwd()))
import saade_v3_core as S
import saade_v4_adaptive as A
P=Path('/home/jovyan/work/audio_real_fake');OUT=P/'results'/'SAADE_AUA_2026_v4_ADAPTIVE_THESIS'
CFG=S.Config(root=str(P/'AUA'),out=str(OUT),scenario_profile='v2_5level')
parents_csv=OUT/'03_parent_splits.csv'
assert parents_csv.is_file(), 'Спочатку запустіть етап 3 основного ноутбука'
parents=pd.read_csv(parents_csv)
queue=A.generate_tts_job_queue(parents,CFG,max_real=400)
print('Черга на транскрибований REAL:',len(queue))
display(queue.head(10))
print('CSV для перевірки:',OUT/'V4_00_TTS_GENERATION_QUEUE.csv')

## Перед генерацією вручну перевірте CSV

Для обраних рядків заповніть `tts_status=APPROVED`, `tts_model_id`, `tts_voice_id`. Потрібні **різні голоси для train / validation / hidden_test**; повторне використання одного голосу між вибірками викличе помилку.

Встановіть Piper CLI і локальні українські `.onnx` TTS-моделі (відповідно до їхніх ліцензій). Нижче приклад адаптера. Система **не встановлює і не завантажує** невідомі моделі автоматично.

In [ ]:
GENERATE=False  # змініть на True лише після ручної перевірки черги та моделей
MODEL_BY_VOICE={
    # 'voice_train_1':'/home/jovyan/work/tts_models/voice_train_1.onnx',
    # 'voice_valid_1':'/home/jovyan/work/tts_models/voice_valid_1.onnx',
    # 'voice_test_1':'/home/jovyan/work/tts_models/voice_test_1.onnx',
}
if GENERATE:
    receipts=A.synthesize_piper_reviewed(OUT/'V4_00_TTS_GENERATION_QUEUE.csv',CFG,MODEL_BY_VOICE)
    display(receipts.head())
    print('TTS рецепти:',OUT/'V4_00_PIPER_GENERATION_RECEIPTS.csv')
else:
    print('Генерація вимкнена; це не помилка. Без receipts режим matched_tts недоступний.')

Після фактичної генерації: у **новому експерименті** встановіть `SOURCE_MODE='matched_tts'`; для такого протоколу рекомендовано використовувати окремий каталог результатів, а не перезаписувати попередні validation/hidden_test.